In [ ]:
# Core
import os
import numpy as np
import pandas as pd

# MATLAB file loading
from scipy.io import loadmat

# Stats
from scipy.stats import ttest_rel
import pingouin as pg

# Mixed models
import statsmodels.formula.api as smf

# Bayesian
import pymc as pm
import arviz as az

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Display
pd.set_option('display.max_rows', 100)

In [ ]:
# Root directory (same as MATLAB scripts)
root_dir = "/rds/projects/f/fernandd-fingerprint-tdcs-fmri/Gugan_Testing/Brain_Parcellation/Out"

# Subdirectories
baseline_dir = os.path.join(root_dir, "Identifiability_results")
within_dir   = os.path.join(root_dir, "Immediate_WithinPolarity_Identifiability")
postpost_dir = os.path.join(root_dir, "PostPost_Identifiability_AcrossConditions")

In [ ]:
def load_identifiability_file(filepath, condition, day=None, comparison=None):
    data = loadmat(filepath)

    Iself = data.get("Iself", []).flatten()
    
    # Handle both naming conventions
    if "Iothers_sub" in data:
        Iothers = data["Iothers_sub"].flatten()
    else:
        Iothers = data["Iothers"].flatten()

    subIDs = data.get("subIDs", np.arange(len(Iself))).flatten()

    df = pd.DataFrame({
        "subject": subIDs,
        "Iself": Iself,
        "Iothers": Iothers,
        "condition": condition,
        "day": day,
        "comparison": comparison
    })

    df["Idiff"] = df["Iself"] - df["Iothers"]
    return df

In [ ]:
baseline_files = {
    "anodal":   "Identifiability_anodal.mat",
    "cathodal": "Identifiability_cathodal.mat",
    "sham":     "Identifiability_sham.mat"
}

df_baseline = pd.concat([
    load_identifiability_file(os.path.join(baseline_dir, f), cond, day="baseline")
    for cond, f in baseline_files.items()
])

In [ ]:
within_files = [
    ("anodal", "day1", "Identifiability_Anodal_Day1_PRE–POST.mat"),
    ("anodal", "day5", "Identifiability_Anodal_Day5_PRE–POST.mat"),
    ("cathodal", "day1", "Identifiability_Cathodal_Day1_PRE–POST.mat"),
    ("cathodal", "day5", "Identifiability_Cathodal_Day5_PRE–POST.mat"),
    ("sham", "day1", "Identifiability_Sham_Day1_PRE–POST.mat"),
    ("sham", "day5", "Identifiability_Sham_Day5_PRE–POST.mat"),
]

df_within = pd.concat([
    load_identifiability_file(os.path.join(within_dir, f), cond, day=day)
    for cond, day, f in within_files
])

In [ ]:
post_files = [
    ("anodal_vs_cathodal", "Identifiability_anodal_vs_cathodal.mat"),
    ("anodal_vs_sham", "Identifiability_anodal_vs_sham.mat"),
    ("cathodal_vs_sham", "Identifiability_cathodal_vs_sham.mat")
]

df_post = pd.concat([
    load_identifiability_file(os.path.join(postpost_dir, f), "comparison", comparison=comp)
    for comp, f in post_files
])

In [ ]:
results_ttest = []

for cond in df_within["condition"].unique():
    for day in df_within["day"].unique():

        subset = df_within[(df_within.condition == cond) & (df_within.day == day)]

        t, p = ttest_rel(subset["Iself"], subset["Iothers"])

        results_ttest.append({
            "condition": cond,
            "day": day,
            "t": t,
            "p": p
        })

df_ttest = pd.DataFrame(results_ttest)
df_ttest

In [ ]:
aov = pg.rm_anova(
    data=df_within,
    dv="Idiff",
    within=["condition", "day"],
    subject="subject",
    detailed=True
)

aov

In [ ]:
model = smf.mixedlm(
    "Idiff ~ condition * day",
    data=df_within,
    groups=df_within["subject"]
)

result = model.fit()
print(result.summary())

In [ ]:
# Encode categorical variables
df_within["cond_code"] = df_within["condition"].astype("category").cat.codes
df_within["day_code"]  = df_within["day"].astype("category").cat.codes

with pm.Model() as model:

    mu = pm.Normal("mu", 0, 1)

    beta_cond = pm.Normal("beta_cond", 0, 1)
    beta_day  = pm.Normal("beta_day", 0, 1)
    beta_int  = pm.Normal("beta_int", 0, 1)

    sigma = pm.HalfNormal("sigma", 1)

    mu_est = (
        mu
        + beta_cond * df_within["cond_code"].values
        + beta_day * df_within["day_code"].values
        + beta_int * df_within["cond_code"].values * df_within["day_code"].values
    )

    y_obs = pm.Normal("y_obs", mu_est, sigma, observed=df_within["Idiff"].values)

    trace = pm.sample(2000, tune=1000, cores=2, target_accept=0.9)

In [ ]:
summary = az.summary(trace, hdi_prob=0.95)
summary

In [ ]:
posterior = trace.posterior

prob_positive = (posterior["beta_cond"] > 0).mean().values
print(f"P(condition effect > 0) = {prob_positive:.3f}")

In [ ]:
az.plot_posterior(trace, var_names=["beta_cond", "beta_day", "beta_int"])
plt.show()

In [ ]:
sns.boxplot(data=df_within, x="condition", y="Idiff", hue="day")
plt.title("Identifiability Difference (Idiff)")
plt.show()